In [1]:
# My benchmark is essentially answering: 
#   For ERNIE sentiment classification at batch size 1 and sequence length of 64,\
#   how much can I improve inference on Graviton3?"
#
# Batch size is the number of input samples processed together in one inference call.

# The model was fine-tuned for one epoch to establish a task-specific sentiment-classification checkpoint;\
# the project focuses on inference optimization rather than maximizing training performance.

### 1. Setup
<hr>

In [2]:
# --- Imports ---
import sys
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments
)
import statistics
import platform
import sys
from pathlib import Path

print(sys.executable)

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

/Users/bhuvanesh/venvs/jlab/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/bhuvanesh/venvs/jlab/bin/python3.11


In [7]:
# --- Experiment Configuration ---

MODEL_NAME = "nghuyong/ernie-3.0-base-zh"

# Model / input
NUM_LABELS = 2
MAX_LENGTH = 64
BATCH_SIZE = 1

# Benchmark
WARMUP_RUNS = 10
NUM_RUNS = 100

# CPU thread tuning
THREAD_COUNTS = (1, 2, 4)  # AWS c7g.xlarge: 4 vCPUs
INTER_THREADS = 1

# Fine-tuning
RUN_FINETUNING = False

In [3]:
# --- Device setup ---
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print(f"Device: {device}")
print(platform.processor())
print(platform.machine())

Device: cpu
arm
arm64


### 2. Load Dataset
<hr>

In [5]:
from src.utils.load_dataset import load_dataset

train_df = load_dataset("train")
dev_df   = load_dataset("dev")
test_df  = load_dataset("test")

# print(len(train_df)) # 9146
# print(len(dev_df))   # 1200
# print(len(test_df))  # 1200
# print(train_df['label'].unique()) # [1 0]
# print(dev_df['label'].unique())   # [1 0]


# test_df = load_dataset("test") # Ignored because there are no labels.
# Use the evaluation dataset (dev_df) for testing instead.

9146
1200
1200
[1 0]
[1 0]


### 3. Load Ernie
<hr>

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

model = model.to(device)

Loading weights: 100%|█████████████████████████████████████| 200/200 [00:00<00:00, 20421.17it/s]
[transformers] ErnieForSequenceClassification LOAD REPORT from: nghuyong/ernie-3.0-base-zh
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 4. Fine-tune ERNIE
<hr>

In [6]:
class ChnSentiCorpDataset(Dataset):

    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text_a"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": self.labels[idx],
        }

In [7]:
train_dataset = ChnSentiCorpDataset(
    train_df, 
    tokenizer,
)

dev_dataset = ChnSentiCorpDataset(
    dev_df, 
    tokenizer,
)

In [9]:
if RUN_FINETUNING:

    from src.evaluation.metrics import compute_metrics

    training_args = TrainingArguments(
        output_dir="../models/checkpoints",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        gradient_accumulation_steps=8,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # save the fine-tuned PyTorch models
    trainer.save_model("../models/ernie-finetuned")
    tokenizer.save_pretrained("../models/ernie-finetuned")

/Users/bhuvanesh/venvs/jlab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.122685,0.260258,0.906667,0.901503,0.910624,0.906040


Writing model shards: 100%|███████████████████████████████████████| 1/1 [00:00<00:00,  1.41it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['ernie.embeddings.LayerNorm.weight', 'ernie.embeddings.LayerNorm.bias', 'ernie.encoder.layer.0.attention.output.LayerNorm.weight', 'ernie.encoder.layer.0.attention.output.LayerNorm.bias', 'ernie.encoder.layer.0.output.LayerNorm.weight', 'ernie.encoder.layer.0.output.LayerNorm.bias', 'ernie.encoder.layer.1.attention.output.LayerNorm.weight', 'ernie.encoder.layer.1.attention.output.LayerNorm.bias', 'ernie.encoder.layer.1.output.LayerNorm.weight', 'ernie.encoder.layer.1.output.LayerNorm.bias', 'ernie.encoder.layer.2.attention.output.LayerNorm.weight', 'ernie.encoder.layer.2.attention.output.LayerNorm.bias', 'ernie.encoder.layer.2.output.LayerNorm.weight', 'ernie.encoder.layer.2.output.LayerNorm.bias', 'ernie.encoder.layer.3.attention.output.LayerNorm.weight', 'ernie.encoder.layer.3.attention.output.LayerNorm.bias', 'ernie.

The OrderedVocab you are attempting to save contains holes for indices [12084], your vocabulary could be corrupted!


In [14]:
# eval_results = trainer.evaluate()

# print("=" * 50)
# print("Fine-tuned ERNIE Quality")
# print("=" * 50)

# print(f"Accuracy : {eval_results['eval_accuracy'] * 100:.2f}%")
# print(f"Precision: {eval_results['eval_precision'] * 100:.2f}%")
# print(f"Recall   : {eval_results['eval_recall'] * 100:.2f}%")
# print(f"F1       : {eval_results['eval_f1'] * 100:.2f}%")

/Users/bhuvanesh/venvs/jlab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
2.122685,0.260258,1,0.906667,0.901503,0.910624,0.906040


Fine-tuned ERNIE Quality
Accuracy : 90.67%
Precision: 90.15%
Recall   : 91.06%
F1       : 90.60%


### 5. Evaluate Fine-tuned Model + PyTorch Baseline Benchmark
<hr>

In [10]:
from src.benchmark.benchmark_pytorch import benchmark_pytorch

FINETUNED_MODEL_PATH = "../models/ernie-finetuned"

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_PATH)

# Load fine-tuned model
model = AutoModelForSequenceClassification.from_pretrained(
    FINETUNED_MODEL_PATH
)

# Select benchmark sample
text = dev_df["text_a"].iloc[0]

# Benchmark PyTorch inference
pytorch_results = benchmark_pytorch(
    model=model,
    tokenizer=tokenizer,
    text=text,
    warmup_runs=WARMUP_RUNS,
    num_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
)


# Notebook
#     load model
#     choose input
#         │
#         ▼
# `benchmark_pytorch()`
#     `model.eval()` -> changes layer behaviour such as dropout
#     tokenize
#     `torch.inference_mode()` -> disables autograd-related work 
#     warmup
#     benchmark
#     metrics

Loading weights: 100%|██████████████████████████████████████| 202/202 [00:00<00:00, 7427.52it/s]


Input shape: torch.Size([1, 64])
PyTorch Benchmark
Runs               : 100
Average latency    : 40.37 ms
Median latency     : 38.27 ms
P95 latency        : 45.96 ms
P99 latency        : 70.45 ms
Minimum latency    : 34.78 ms
Maximum latency    : 87.45 ms
Throughput         : 24.77 samples/sec


In [11]:
# from src.profiling.profile_pytorch import profile_pytorch

# profile_pytorch(
#     model=model,
#     tokenizer=tokenizer,
#     text=text,
#     warmup_runs=10,
#     max_length=64,
#     row_limit=20,
# )

### 6. Export to ONNX
<hr>

In [12]:
from src.optimization.export_onnx import export_onnx, validate_onnx

onnx_model_path = "../models/ernie_finetuned.onnx"

export_onnx(
    model=model,
    tokenizer=tokenizer,
    text=text,
    output_path=onnx_model_path,
    max_length=MAX_LENGTH,
    opset_version=17,
)

validate_onnx(onnx_model_path)

/Users/bhuvanesh/Desktop/Jupyter/LLM/arm-ai-optimization/src/optimization/export_onnx.py:33: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0814 11:57:50.055000 3168 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ErnieForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ErnieForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/opt/homebrew/Cellar/python@3.11/3.11.14_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/Users/bhuvanesh/venvs/jlab/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


ONNX model exported to: ../models/ernie_finetuned.onnx
ONNX model is valid!


### 7. ONNX Benchmark
<hr>

In [13]:
from src.benchmark.benchmark_onnx import benchmark_onnx

onnx_result = benchmark_onnx(
    model_path=onnx_model_path,
    tokenizer=tokenizer,
    text=text,
    warmup_runs=WARMUP_RUNS,
    num_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
)

Providers: ['CPUExecutionProvider']
Graph optimization: GraphOptimizationLevel.ORT_ENABLE_ALL
Intra-op threads: 0
Inter-op threads: 0
Execution mode: ExecutionMode.ORT_SEQUENTIAL
Input shape: (1, 64)
ONNX Runtime Benchmark
Runs               : 100
Average latency    : 37.62 ms
Median latency     : 33.82 ms
P95 latency        : 52.35 ms
P99 latency        : 116.43 ms
Minimum latency    : 33.48 ms
Maximum latency    : 145.83 ms
Throughput         : 26.58 samples/sec


### 8. Graph Optimization
<hr>

In [18]:
from src.optimization.optimize_onnx import benchmark_optimization_levels

optimization_results, best_optimization_level = benchmark_optimization_levels(
    model_path="../models/ernie_finetuned.onnx",
    tokenizer=tokenizer,
    text=text,
    warmup_runs=WARMUP_RUNS,
    num_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
)

print(best_optimization_level)

Disable    : 39.72 ms
Basic      : 36.70 ms
Extended   : 36.13 ms
All        : 36.47 ms
GraphOptimizationLevel.ORT_ENABLE_EXTENDED


### 9. Thread Tuning
<hr>

In [19]:
from src.optimization.tune_threads import tune_threads

thread_results, best_thread_config = tune_threads(
    model_path="../models/ernie_finetuned.onnx",
    tokenizer=tokenizer,
    text=text,
    thread_counts=THREAD_COUNTS,     
    inter_threads=INTER_THREADS,
    warmup_runs=WARMUP_RUNS,
    num_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
    optimization_level=best_optimization_level,
)

In [20]:
print("=" * 75)
print("Thread Tuning Summary")
print("=" * 75)

print(
    f"{'Intra':<10}"
    f"{'Avg (ms)':<12}"
    f"{'Median':<12}"
    f"{'P95':<12}"
    f"{'Throughput':<15}"
)

for result in thread_results:
    print(
        f"{result['intra_threads']:<10}"
        f"{result['average']:<12.2f}"
        f"{result['median']:<12.2f}"
        f"{result['p95']:<12.2f}"
        f"{result['throughput']:<15.2f}"
    )

Thread Tuning Summary
Intra     Avg (ms)    Median      P95         Throughput     
1         117.24      113.48      126.44      8.53           
2         65.48       59.76       102.05      15.27          
4         39.93       33.89       62.58       25.04          


In [21]:
print()
print("Best configuration")
print("-" * 40)

print(
    f"Intra-op threads : "
    f"{best_thread_config['intra_threads']}"
)

print(
    f"Inter-op threads : "
    f"{best_thread_config['inter_threads']}"
)

print(
    f"Median latency   : "
    f"{best_thread_config['median']:.2f} ms"
)

print(
    f"Average latency  : "
    f"{best_thread_config['average']:.2f} ms"
)

print(
    f"Throughput       : "
    f"{best_thread_config['throughput']:.2f} samples/sec"
)


Best configuration
----------------------------------------
Intra-op threads : 4
Inter-op threads : 1
Median latency   : 33.89 ms
Average latency  : 39.93 ms
Throughput       : 25.04 samples/sec


### Quantize the FP32 ONNX model
<hr>

In [22]:
from src.optimization.quantize import quantize_model

quantize_model(
    input_model_path="../models/ernie_finetuned.onnx",
    relaxed_model_path="../models/ernie_finetuned_relaxed.onnx",
    output_model_path="../models/ernie_finetuned_int8.onnx",
)

INT8 model written to: ../models/ernie_finetuned_int8.onnx


In [23]:
from pathlib import Path

# Helper
def model_size_mb(*paths):
    total_bytes = sum(
        Path(path).stat().st_size
        for path in paths
        if Path(path).exists()
    )

    return total_bytes / (1024 ** 2)

### Compare FP32 vs INT8 model size
<hr>

In [24]:
fp32_size = model_size_mb(
    "../models/ernie_finetuned.onnx",
    "../models/ernie_finetuned.onnx.data",
)

int8_size = model_size_mb(
    "../models/ernie_finetuned_int8.onnx",
    "../models/ernie_finetuned_int8.onnx.data",
)

print(f"FP32 size : {fp32_size:.2f} MB")
print(f"INT8 size : {int8_size:.2f} MB")
print(f"Reduction : {(1 - int8_size / fp32_size) * 100:.2f}%")

FP32 size : 450.11 MB
INT8 size : 113.39 MB
Reduction : 74.81%


### Evaluate INT8 accuracy on the dev set
<hr>

In [25]:
from src.evaluation.evaluate_onnx import evaluate_onnx
y_true = dev_df["label"].to_numpy()


int8_metrics = evaluate_onnx(
    model_path="../models/ernie_finetuned_int8.onnx",
    tokenizer=tokenizer,
    texts=dev_df["text_a"],
    labels=y_true,
    max_length=MAX_LENGTH,
)

print(int8_metrics)

{'accuracy': 0.8908333333333334, 'precision': 0.8553846153846154, 'recall': 0.9376053962900506, 'f1': 0.8946098149637972}


In [26]:
fp32_metrics = evaluate_onnx(
    model_path="../models/ernie_finetuned.onnx",
    tokenizer=tokenizer,
    texts=dev_df["text_a"],
    labels=y_true,
    max_length=MAX_LENGTH,
)

print(fp32_metrics)

{'accuracy': 0.9116666666666666, 'precision': 0.9078726968174204, 'recall': 0.9139966273187183, 'f1': 0.9109243697478991}


In [27]:
print(f"Accuracy change:  {(int8_metrics['accuracy'] - fp32_metrics['accuracy']) * 100:+.2f}%")
print(f"Precision change: {(int8_metrics['precision'] - fp32_metrics['precision']) * 100:+.2f}%")
print(f"Recall change:    {(int8_metrics['recall'] - fp32_metrics['recall']) * 100:+.2f}%")
print(f"F1 change:        {(int8_metrics['f1'] - fp32_metrics['f1']) * 100:+.2f}%")

Accuracy change:  -2.08%
Precision change: -5.25%
Recall change:    +2.36%
F1 change:        -1.63%


In [28]:
int8_optimization_results, best_int8_optimization_level = (
    benchmark_optimization_levels(
        model_path="../models/ernie_finetuned_int8.onnx",
        tokenizer=tokenizer,
        text=text,
    )
)

Disable    : 18.96 ms
Basic      : 15.38 ms
Extended   : 13.14 ms
All        : 13.43 ms


In [29]:
int8_thread_results, best_int8_thread_config = tune_threads(
    model_path="../models/ernie_finetuned_int8.onnx",
    tokenizer=tokenizer,
    text=text,
    thread_counts=THREAD_COUNTS,
    inter_threads=INTER_THREADS,
    warmup_runs=WARMUP_RUNS,
    num_runs=NUM_RUNS,
    max_length=MAX_LENGTH,
    optimization_level=best_int8_optimization_level,
)

In [30]:
print("=" * 75)
print("Thread Tuning Summary")
print("=" * 75)

print(
    f"{'Intra':<10}"
    f"{'Avg (ms)':<12}"
    f"{'Median':<12}"
    f"{'P95':<12}"
    f"{'Throughput':<15}"
)

for result in int8_thread_results:
    print(
        f"{result['intra_threads']:<10}"
        f"{result['average']:<12.2f}"
        f"{result['median']:<12.2f}"
        f"{result['p95']:<12.2f}"
        f"{result['throughput']:<15.2f}"
    )

Thread Tuning Summary
Intra     Avg (ms)    Median      P95         Throughput     
1         39.58       38.57       40.95       25.26          
2         21.28       21.27       21.34       46.99          
4         13.62       12.79       13.64       73.41          


In [31]:
print()
print("Best configuration")
print("-" * 40)

print(
    f"Intra-op threads : "
    f"{best_int8_thread_config['intra_threads']}"
)

print(
    f"Inter-op threads : "
    f"{best_int8_thread_config['inter_threads']}"
)

print(
    f"Median latency   : "
    f"{best_int8_thread_config['median']:.2f} ms"
)

print(
    f"Average latency  : "
    f"{best_int8_thread_config['average']:.2f} ms"
)

print(
    f"Throughput       : "
    f"{best_int8_thread_config['throughput']:.2f} samples/sec"
)


Best configuration
----------------------------------------
Intra-op threads : 4
Inter-op threads : 1
Median latency   : 12.79 ms
Average latency  : 13.62 ms
Throughput       : 73.41 samples/sec


In [34]:
print("=" * 55)
print("Model Quality Comparison")
print("=" * 55)

print(
    f"{'Model':<15}"
    f"{'Accuracy':<15}"
    f"{'F1':<15}"
)

print("-" * 55)

print(
    f"{'ONNX FP32':<15}"
    f"{fp32_metrics['accuracy'] * 100:<15.2f}"
    f"{fp32_metrics['f1'] * 100:<15.2f}"
)

print(
    f"{'ONNX INT8':<15}"
    f"{int8_metrics['accuracy'] * 100:<15.2f}"
    f"{int8_metrics['f1'] * 100:<15.2f}"
)

print("=" * 55)

print(
    f"Accuracy change : "
    f"{(int8_metrics['accuracy'] - fp32_metrics['accuracy']) * 100:+.2f} pp"
)

print(
    f"F1 change       : "
    f"{(int8_metrics['f1'] - fp32_metrics['f1']) * 100:+.2f} pp"
)

Model Quality Comparison
Model          Accuracy       F1             
-------------------------------------------------------
ONNX FP32      91.17          91.09          
ONNX INT8      89.08          89.46          
Accuracy change : -2.08 pp
F1 change       : -1.63 pp
